# Training Diagnostics Dashboard

Load an MLflow run by `run_id` and display:
1. Training curves (losses, rewards)
2. Trading performance metrics
3. Action distributions & heatmaps
4. Per-asset PnL breakdown
5. TI correlations & regime analysis

**Usage:** Set `RUN_ID` below, then Run All.

In [ ]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Configure
TRACKING_URI = "mlruns"
RUN_ID = "<PASTE_RUN_ID_HERE>"

mlflow.set_tracking_uri(TRACKING_URI)
client = mlflow.tracking.MlflowClient()

In [ ]:
# Load run
run = client.get_run(RUN_ID)
metrics = run.data.metrics
params = run.data.params

print(f"Run: {run.info.run_name}")
print(f"Status: {run.info.status}")
print(f"\nParameters:")
for k, v in sorted(params.items()):
    print(f"  {k}: {v}")

print(f"\nKey Metrics:")
perf_keys = [k for k in sorted(metrics.keys()) if k.startswith("perf/")]
for k in perf_keys:
    print(f"  {k}: {metrics[k]:.6f}")

In [ ]:
# Training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, metric_key, title in [
    (axes[0, 0], "train/critic_loss", "Critic Loss"),
    (axes[0, 1], "train/actor_loss", "Actor Loss"),
    (axes[1, 0], "episode/reward", "Episode Reward"),
    (axes[1, 1], "episode/portfolio_value", "Episode Portfolio Value"),
]:
    history = client.get_metric_history(RUN_ID, metric_key)
    if history:
        steps = [h.step for h in history]
        values = [h.value for h in history]
        ax.plot(steps, values, alpha=0.7)
        ax.set_title(title)
        ax.set_xlabel("Step")
        ax.grid(True, alpha=0.3)
    else:
        ax.set_title(f"{title} (no data)")

fig.tight_layout()
plt.show()

In [ ]:
# Load equity curve artifact
artifacts_dir = client.download_artifacts(RUN_ID, "evaluation")
eq_path = Path(artifacts_dir) / "equity_curve.csv"

if eq_path.exists():
    eq_df = pd.read_csv(eq_path)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(eq_df["step"], eq_df["portfolio_value"], color="blue", alpha=0.8)
    ax.set_title("Equity Curve")
    ax.set_xlabel("Step")
    ax.set_ylabel("Portfolio Value")
    ax.axhline(y=eq_df["portfolio_value"].iloc[0], color="gray", linestyle="--", alpha=0.5)
    ax.grid(True, alpha=0.3)
    plt.show()
else:
    print("No equity curve artifact found.")

In [ ]:
# Per-asset breakdown
asset_keys = [k for k in sorted(metrics.keys()) if k.startswith("asset/")]
if asset_keys:
    print("Per-Asset Metrics:")
    for k in asset_keys:
        print(f"  {k}: {metrics[k]:.4f}")
else:
    print("No per-asset metrics found.")

In [ ]:
# Regime analysis
regime_keys = [k for k in sorted(metrics.keys()) if k.startswith("regime/")]
if regime_keys:
    print("Regime Analysis:")
    for k in regime_keys:
        print(f"  {k}: {metrics[k]:.4f}")
else:
    print("No regime metrics found.")

In [ ]:
# Action distribution over time
actions_mean_history = client.get_metric_history(RUN_ID, "actions/mean")
actions_std_history = client.get_metric_history(RUN_ID, "actions/std")

if actions_mean_history:
    fig, ax = plt.subplots(figsize=(12, 5))
    steps = [h.step for h in actions_mean_history]
    means = [h.value for h in actions_mean_history]
    ax.plot(steps, means, label="Action Mean", alpha=0.8)
    if actions_std_history:
        stds = [h.value for h in actions_std_history]
        ax.fill_between(steps, 
                        [m - s for m, s in zip(means, stds)],
                        [m + s for m, s in zip(means, stds)],
                        alpha=0.2)
    ax.set_title("Action Distribution Over Training")
    ax.set_xlabel("Step")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.show()
else:
    print("No action distribution data found.")

## Diagnosis Template

Fill in after reviewing the plots above:

1. **Is the critic converging?** (Loss should decrease over time)
2. **Is the actor learning?** (Reward should increase, or at least stabilize)
3. **Action diversity:** Are actions clustered at 0 or 1? (Bad — agent isn't exploring)
4. **Per-asset performance:** Which assets are profitable? Any consistently losing?
5. **Regime sensitivity:** Does the agent perform differently in up/down/sideways markets?
6. **Trade timing:** Is it buying below SMA20 and selling above? (Smart timing indicator)
7. **Cost impact:** What % of returns are eaten by trading costs?

### Next Steps
- [ ] Adjust hyperparameters based on findings
- [ ] Consider curriculum learning if regime performance is uneven
- [ ] Check for reward shaping issues if skewness is extreme